# 01 - Fetch NGS CORS RINEX for the Gannon window

Walks through identifying the target stations, fetching daily RINEX observation files for May 8-14, 2024 (Gannon storm window), and sanity-checking each station's header.

**Inputs:** NGS CORS public archive (`https://geodesy.noaa.gov/corsdata/rinex/`).
**Outputs:** `data/cors/2024/<doy>/<station>/<station><doy>0.24d.gz` (gitignored cache).

Run order: 01 → 02 → 03.

## 1. Station catalog

The catalog is committed at `src/gannon_analysis/stations.py`. It was assembled by probing the NGS CORS archive for stations with day-of-year 131 (2024-05-10) data live on the day this analysis was first run, then extracting each station's published ITRF2014 truth coordinates from the `APPROX POSITION XYZ` header field.

In [ ]:
from gannon_analysis.stations import CORS_STATIONS, stations_by_state

for state in ("IA", "IL", "IN", "OH"):
    rows = stations_by_state(state)
    print(f"{state}: {len(rows):2d} stations  | ", ", ".join(s.station_id for s in rows))
print(f"\nTotal corridor stations: {len(CORS_STATIONS)}")

## 2. Fetch RINEX for one station-day (smoke test)

`fetch_rinex` caches files locally and is polite (250 ms between requests by default). Re-running is idempotent.

In [ ]:
from datetime import date
from pathlib import Path

from gannon_analysis.fetch import fetch_rinex

path = fetch_rinex("iaal", date(2024, 5, 10), Path("../data"))
print(f"Cached: {path}")
print(f"Size: {path.stat().st_size / 1024:.1f} KB")

## 3. Sanity-check the RINEX header

In [ ]:
import georinex as gr

hdr = gr.rinexheader(str(path))
for key in (
    "MARKER NAME",
    "REC # / TYPE / VERS",
    "ANT # / TYPE",
    "APPROX POSITION XYZ",
    "interval",
    "systems",
):
    val = hdr.get(key, hdr.get(key.lower()))
    print(f"{key:>30} : {val}")

## 4. Fetch the full corridor for the storm window

Cold-cache estimate: ~5 min for all 25 stations × 7 days (175 files at ~500 KB each). Warm-cache: <2 s.

In [ ]:
import httpx

from gannon_analysis.fetch import fetch_window
from gannon_analysis.stations import stations_in_corridor

data_dir = Path("../data")
with httpx.Client(timeout=60.0) as client:
    for station in stations_in_corridor():
        files = fetch_window(station, data_dir, client=client)
        print(f"{station.station_id} ({station.state}) - {len(files)} days cached")

## Next

Proceed to `02-positioning-solutions.ipynb` to compute per-station 2D horizontal error and to `03-correlate-swpc.ipynb` for the headline regional aggregate plot.